# HCP-D fMRI time-series and classical metric extraction

This notebook applies the timeseries extraction workflow to HCP Development data. Parameters is defined in `configs/hcpd_timeseries_parameters.jsonc`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    raise FileNotFoundError("Run this notebook from the RISE repository root or pipeline directory")
from rise import (
    build_hcpd_run_inputs,
    extract_dataset,
    load_config,
    prepare_atlases,
    read_subject_ids,
)

### Configuration

Relative paths are resolved from the configuration directory. Edit the JSONC file for persistent defaults, or pass an `overrides` dictionary to `load_config` for a specific run.

In [ ]:
CONFIG_FILE = PROJECT_ROOT / "configs" / "hcpd_timeseries_parameters.jsonc"
config = load_config(CONFIG_FILE)


### Discover HCP-D runs

In [ ]:
subjects = read_subject_ids(*config["subject_files"])
run_inputs = build_hcpd_run_inputs(subjects, config)
print(f"Found {len(run_inputs)} runs for {len(subjects)} subjects.")
run_inputs[:4]

In [ ]:
# Verify how many runs were found for each subject.
run_counts = {}
for item in run_inputs:
    run_counts[item["id"]] = run_counts.get(item["id"], 0) + 1
missing_runs = {subject: run_counts.get(subject, 0) for subject in subjects if run_counts.get(subject, 0) != 4}
print(f"Subjects without four runs: {len(missing_runs)}")
list(missing_runs.items())[:10]

### Prepare atlas files

The packaged atlases and the brain mask are resampled once to the HCP-D BOLD resolution and cached under `data/HCPD/atlas_cache`.

In [ ]:
if not run_inputs:
    raise RuntimeError("No HCP-D runs were found. Check dataset_root and subject IDs.")

atlas_cache = prepare_atlases(run_inputs[0]["nii_file"], config, force=False)
print(f"Prepared atlases: {atlas_cache}")

### Run extraction

Outputs are written under `data/HCPD/processed_data/indiv/<subject>/func/<run>/data`.

In [ ]:
results = extract_dataset(
    run_inputs[:4],#Only the first 4 runs were executed as an example
    config,
    n_processes=config["n_processes"],
)
results[:4]